# ENEM Predictor — Treinamento do Modelo Híbrido & Predição de Temas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leonardo-py-dev/enem-aquisicao-de-dados/blob/main/notebooks/enem_predictor_training.ipynb)

Este notebook executa o ciclo completo de Machine Learning do **ENEM Predictor**:
1. **Conexão Direta ao Supabase:** Carrega dados históricos oficiais (1998–2024), notícias do IBGE, proposições da Câmara e artigos científicos.
2. **Engenharia de Atributos:** Calcula ciclos históricos do INEP, vetos de repetição consecutiva e intensidade de debate social.
3. **Fase 1 (Classificador Quantitativo):** Treina modelo **XGBoost** com validação temporal (*Walk-Forward*) e ranqueia os 10 eixos oficiais.
4. **Fase 2 (RAG & Geração Semântica):** Usa a **Google Gemini API (Gemini 2.5 Flash)** para buscar evidências do eixo vencedor e gerar 3 temas candidatos completos com propostas de intervenção nos 5 critérios oficiais da redação nota 1000.

### 1. Instalação das Dependências na Nuvem do Colab
Instala as bibliotecas sem usar nenhum recurso da sua máquina local.

In [ ]:
!pip install -q sqlalchemy "psycopg[binary]" psycopg2-binary pgvector pandas numpy scikit-learn xgboost matplotlib seaborn google-genai

### 2. Configuração de Credenciais Seguras
Configure os segredos `DATABASE_URL` e `GEMINI_API_KEY` na aba de segredos (ícone de chave 🔑) do Colab ou insira diretamente abaixo.

In [ ]:
import os
import getpass

DATABASE_URL = None
GEMINI_API_KEY = None

# 1. Tenta carregar automaticamente dos Segredos permanentes do Google Colab (Ícone 🔑)
try:
    from google.colab import userdata
    try:
        DATABASE_URL = userdata.get('DATABASE_URL')
    except Exception:
        pass
    try:
        GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    except Exception:
        pass
except Exception:
    pass

# 2. Fallback para variáveis de ambiente
if not DATABASE_URL:
    DATABASE_URL = os.environ.get('DATABASE_URL')
if not GEMINI_API_KEY:
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')

# 3. Solicitação manual apenas se não estiver nos Segredos
if DATABASE_URL and GEMINI_API_KEY:
    print("🔒 Credenciais carregadas AUTOMATICAMENTE dos Segredos permanentes do Colab!")
else:
    if not DATABASE_URL:
        DATABASE_URL = getpass.getpass("DATABASE_URL não encontrada nos Segredos. Cole manualmente: ")
    if not GEMINI_API_KEY:
        GEMINI_API_KEY = getpass.getpass("GEMINI_API_KEY não encontrada nos Segredos. Cole manualmente: ")
    print("✅ Credenciais configuradas com sucesso!")

### 3. Conexão ao Supabase e Carga dos Dados Históricos

In [ ]:
from sqlalchemy import create_engine, text
import pandas as pd

# Normaliza o dialeto para psycopg3 caso necessário
db_url = DATABASE_URL.strip()
if db_url.startswith("postgres://"):
    db_url = db_url.replace("postgres://", "postgresql+psycopg://", 1)
elif db_url.startswith("postgresql://") and not db_url.startswith("postgresql+"):
    db_url = db_url.replace("postgresql://", "postgresql+psycopg://", 1)

engine = create_engine(db_url)

with engine.connect() as conn:
    # Carrega histórico oficial do ENEM
    themes_df = pd.read_sql("SELECT year, official_theme, thematic_axis, central_problem FROM historical_themes ORDER BY year ASC;", conn)
    # Carrega os 10 eixos oficiais
    topics_df = pd.read_sql("SELECT id, name, thematic_axis FROM topics ORDER BY id ASC;", conn)
    # Carrega documentos coletados
    docs_df = pd.read_sql("""
        SELECT d.id, d.title, d.published_at, s.name as source_name, t.thematic_axis, dt.relevance_score
        FROM documents d
        LEFT JOIN sources s ON d.source_id = s.id
        LEFT JOIN document_topics dt ON d.id = dt.document_id
        LEFT JOIN topics t ON dt.topic_id = t.id;
    """, conn)

print(f"✅ Carregados {len(themes_df)} temas históricos do ENEM (1998 a 2024)")
print(f"✅ Carregados {len(topics_df)} eixos temáticos cadastrados")
print(f"✅ Carregados {len(docs_df)} registros de documentos/vínculos temáticos")

themes_df.tail(5)

### 4. Engenharia de Atributos (Matriz Temporal de Treino)
Constrói as features ano a ano para cada um dos 10 eixos, respeitando rigorosamente a linha do tempo (sem vazamento de dados futuros).

In [ ]:
history_map = dict(zip(themes_df['year'], themes_df['thematic_axis']))
axes = topics_df['thematic_axis'].unique().tolist()

features_rows = []
max_historical_year = int(themes_df['year'].max())

for target_year in range(2003, max_historical_year + 1):
    actual_axis = history_map.get(target_year)
    prev_axis = history_map.get(target_year - 1)

    for axis in axes:
        past_occurrences = [y for y, ax in history_map.items() if y < target_year and ax == axis]
        years_since_last = (target_year - max(past_occurrences)) if past_occurrences else (target_year - 1998)
        occurred_prev_year = 1 if prev_axis == axis else 0
        occurred_last_3 = sum(1 for y in [target_year - 1, target_year - 2, target_year - 3] if history_map.get(y) == axis)

        total_past_exams = sum(1 for y in history_map.keys() if y < target_year)
        freq_acumulada = (len(past_occurrences) / total_past_exams) if total_past_exams > 0 else 0.0

        # Features de volume social recente
        if not docs_df.empty and 'published_at' in docs_df.columns and docs_df['published_at'].notna().any():
            doc_year = docs_df['published_at'].apply(lambda d: d.year if pd.notna(d) else target_year)
            mask = (doc_year == target_year) & (docs_df['thematic_axis'] == axis)
            vol_social = len(docs_df[mask])
            score_social = docs_df[mask]['relevance_score'].sum() if vol_social > 0 else 0.0
        else:
            vol_social = 0
            score_social = 0.0

        is_target = 1 if actual_axis == axis else 0

        features_rows.append({
            'year': target_year,
            'axis': axis,
            'years_since_last': years_since_last,
            'occurred_previous_year': occurred_prev_year,
            'occurred_last_3_years': occurred_last_3,
            'cumulative_frequency': freq_acumulada,
            'vol_social': vol_social,
            'score_social': score_social,
            'target': is_target,
        })

train_df = pd.DataFrame(features_rows)
print(f"✅ Matriz gerada com {len(train_df)} observações tabularizadas")
train_df.sample(5)

### 5. Treinamento do Modelo XGBoost (Fase 1: Ranking dos 10 Eixos)
Treina o modelo com validação temporal (*Walk-Forward*) e exibe as importâncias das variáveis.

In [ ]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

feature_cols = [
    'years_since_last',
    'occurred_previous_year',
    'occurred_last_3_years',
    'cumulative_frequency',
    'vol_social',
    'score_social'
]

# Treina com dados até 2023 e avalia no ENEM 2024
X_train = train_df[train_df['year'] < 2024][feature_cols]
y_train = train_df[train_df['year'] < 2024]['target']

X_val = train_df[train_df['year'] == 2024][feature_cols]
y_val = train_df[train_df['year'] == 2024]['target']

model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    eval_metric='logloss',
    random_state=42
)
model.fit(X_train, y_train)

# Gráfico de Importância das Features
plt.figure(figsize=(8, 4))
sns.barplot(x=model.feature_importances_, y=feature_cols, palette='viridis')
plt.title('Importância das Features na Decisão do Modelo (XGBoost)')
plt.xlabel('Importância Relativa')
plt.show()

### 6. Predição de Probabilidade para a Próxima Prova
Calcula a probabilidade de cada um dos 10 eixos oficiais para a próxima edição do ENEM.

In [ ]:
# Constrói a linha de features para a próxima edição (calculada dinamicamente)
last_year = int(themes_df['year'].max())
target_next_year = last_year + 1
last_axis = history_map.get(last_year)

pred_rows = []
for axis in axes:
    past = [y for y, ax in history_map.items() if ax == axis]
    years_since = target_next_year - max(past) if past else (target_next_year - 1998)
    occ_prev = 1 if last_axis == axis else 0
    occ_3 = sum(1 for y in [last_year, last_year - 1, last_year - 2] if history_map.get(y) == axis)
    freq = len(past) / len(history_map)

    pred_rows.append({
        'axis': axis,
        'years_since_last': years_since,
        'occurred_previous_year': occ_prev,
        'occurred_last_3_years': occ_3,
        'cumulative_frequency': freq,
        'vol_social': 0,
        'score_social': 0.0,
    })

df_next = pd.DataFrame(pred_rows)
df_next['probability'] = model.predict_proba(df_next[feature_cols])[:, 1]
df_next['prob_percent'] = (df_next['probability'] / df_next['probability'].sum()) * 100
df_next = df_next.sort_values(by='prob_percent', ascending=False).reset_index(drop=True)

# Gráfico de Ranking dos Eixos
plt.figure(figsize=(10, 5))
bars = sns.barplot(data=df_next, x='prob_percent', y='axis', palette='Blues_r')
plt.title(f'ENEM Predictor — Probabilidade dos 10 Eixos Temáticos ({target_next_year})')
plt.xlabel('Probabilidade Estimada (%)')
for p in bars.patches:
    bars.annotate(f'{p.get_width():.1f}%', (p.get_width() + 0.5, p.get_y() + p.get_height() / 2), ha='left', va='center')
plt.xlim(0, max(df_next['prob_percent']) + 5)
plt.show()

print("🏆 TOP 3 EIXOS MAIS PROVÁVEIS:")
for i, row in df_next.head(3).iterrows():
    print(f"{i+1}º Lugar: {row['axis']} — {row['prob_percent']:.1f}% de probabilidade (Última vez que caiu: {target_next_year - row['years_since_last']})")

### 7. Fase 2: RAG & Geração de Temas com Google Gemini 2.5 Flash
Utiliza a API do Gemini para gerar formulações oficiais do tema e propostas de intervenção completas com os 5 elementos (Agente, Ação, Meio/Modo, Efeito e Detalhamento).

In [ ]:
from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)
top_axis = df_next.iloc[0]['axis']
second_axis = df_next.iloc[1]['axis']

prompt = f"""
Você é o especialista chefe em formulação de temas de redação do Exame Nacional do Ensino Médio (ENEM) no INEP.

O nosso modelo estatístico apontou que o eixo temático com MAIOR probabilidade para a próxima prova é: "{top_axis}", seguido por: "{second_axis}".

Com base nas diretrizes da banca examinadora do ENEM e nos desafios contemporâneos da sociedade brasileira:
1. Formule 3 opções de temas de redação realistas para o eixo "{top_axis}", seguindo fielmente a sintaxe da banca (ex: "Desafios para...", "Caminhos para...", "O estigma associado a...", "A persistência de...").
2. Para cada uma das 3 opções, apresente:
   - Problema Social Central.
   - Ideia para os Textos Motivadores (notícias, estatísticas ou marcos legais).
   - Proposta de Intervenção Social Completa e Detalhada contemplando explicitamente os 5 elementos de avaliação da Competência 5:
     * Agente (Quem executa)
     * Ação (O que será feito)
     * Meio/Modo (Como será feito)
     * Efeito/Finalidade (Para que será feito)
     * Detalhamento (Explicação adicional de um dos elementos)

Estruture a resposta em Markdown com formatação visual clara e elegante.
"""

print(f"🤖 Consultando Gemini 2.5 Flash para formulação de temas para o eixo vencedor: '{top_axis}'...")
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt,
)

from IPython.display import display, Markdown
display(Markdown(response.text))